# Credit Card Churn Analyzer — SHAP Extension
### Section 11: Explainability with SHAP Values

---

**Why SHAP?**

Feature importances tell you what the model cares about globally.  
SHAP tells you *why a specific customer is at risk* — in plain terms a business can act on.

This is the difference between:
- ❌ "Inactivity is an important feature"
- ✅ "Customer CUST_00042 is high-risk primarily because they've been inactive for 9 months and haven't redeemed rewards in 6 months"

That second version drives a retention action. The first one doesn't.

---

**What this section covers:**
1. Install and compute SHAP values
2. Global summary plot — which features drive churn across all customers
3. Waterfall chart — explain a single high-risk customer
4. Top 3 risk reasons per customer — exportable table for ops teams

## 11.0 — Prerequisites

> ⚠️ **Run the full base notebook first** (`credit_card_churn_analyzer.ipynb`) so that
> `model`, `X_train`, `X_test`, `X`, `df_scored`, and `FEATURES` are all in memory.
> Then run this notebook in the same kernel session — or paste these cells directly
> after Section 10 in the base notebook.

Install SHAP if needed:

In [ ]:
# Install SHAP (run once)
# !pip install shap

import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

shap.__version__

## 11.1 — Compute SHAP Values

`TreeExplainer` is purpose-built for tree-based models like Random Forest.  
It's exact (not an approximation) and runs fast on tabular data.

In [ ]:
# Friendly display names (same as base notebook)
name_map = {
    'months_inactive_last_12m':   'Months Inactive (12M)',
    'num_transactions_last_3m':   'Transactions (3M)',
    'avg_monthly_spend':          'Avg Monthly Spend',
    'reward_redemptions_last_6m': 'Reward Redemptions (6M)',
    'num_products':               'Number of Products',
    'utilization_rate':           'Utilization Rate',
    'credit_limit':               'Credit Limit',
    'tenure_months':              'Tenure (Months)',
    'contacted_support_last_6m':  'Support Contacts (6M)',
    'income_segment_enc':         'Income Segment',
    'card_tier_enc':              'Card Tier',
}

# Use a background sample for efficiency (200 rows is enough)
background = shap.sample(X_train, 200, random_state=42)

explainer   = shap.TreeExplainer(model, background)
shap_values = explainer(X_test)   # returns Explanation object

# We want SHAP values for class 1 (churn)
# shap_values[:, :, 1] → shape (n_test_samples, n_features)
shap_churn = shap_values[:, :, 1]

# Rename features for clean charts
shap_churn.feature_names = [name_map.get(f, f) for f in FEATURES]

print(f'SHAP values computed for {shap_churn.shape[0]} test customers across {shap_churn.shape[1]} features.')

## 11.2 — Global Summary Plot

This plot shows:
- **Which features matter most** (y-axis order = mean absolute SHAP)
- **Direction of impact** (red = increases churn risk, blue = decreases it)
- **Feature value distribution** (color intensity = high vs low feature value)

This is the chart you lead with in a presentation or portfolio README.

In [ ]:
plt.figure(figsize=(10, 6))
shap.plots.beeswarm(
    shap_churn,
    max_display=11,
    show=False,
)
plt.title('SHAP Summary — Global Churn Drivers', fontweight='bold', fontsize=13, pad=12)
plt.xlabel('SHAP Value (impact on churn probability)', fontsize=10)
plt.tight_layout()
plt.savefig('shap_summary_beeswarm.png', bbox_inches='tight')
plt.show()
print('Saved: shap_summary_beeswarm.png')

**How to read this chart:**

- Each dot = one customer in the test set
- **Red dots** = high feature value for that customer
- **Blue dots** = low feature value
- Dots pushed **right** = that feature *increased* their churn risk
- Dots pushed **left** = that feature *reduced* their churn risk

Example read: High inactivity (red dots, top row, pushed right) = strong churn signal.  
More products (red dots, pushed left) = strong retention signal.

## 11.3 — Waterfall Chart: Explain One High-Risk Customer

Pick the highest-risk customer from the test set and show exactly what's driving their score.  
This is the output a retention manager actually uses.

In [ ]:
# Find the index of the highest-risk customer in X_test
y_proba_test = model.predict_proba(X_test)[:, 1]
high_risk_idx = int(np.argmax(y_proba_test))

# Pull their raw feature values for context
customer_features = X_test.iloc[high_risk_idx]
churn_prob        = y_proba_test[high_risk_idx]

print(f'Highest-risk customer in test set')
print(f'Churn Probability : {churn_prob:.1%}')
print(f'Actual Churned    : {y_test.iloc[high_risk_idx]}')
print()
print('Feature values:')
for feat in FEATURES:
    label = name_map.get(feat, feat)
    print(f'  {label:<35} {customer_features[feat]}')

In [ ]:
# Waterfall plot — shows how each feature pushed the score up or down from baseline
plt.figure(figsize=(10, 6))
shap.plots.waterfall(
    shap_churn[high_risk_idx],
    max_display=11,
    show=False,
)
plt.title(
    f'SHAP Waterfall — Why Is This Customer High Risk?\nChurn Probability: {churn_prob:.1%}',
    fontweight='bold', fontsize=12, pad=12
)
plt.tight_layout()
plt.savefig('shap_waterfall_high_risk.png', bbox_inches='tight')
plt.show()
print('Saved: shap_waterfall_high_risk.png')

**How to read this chart:**

- **E[f(x)]** = baseline churn probability (average across all customers)
- **f(x)** = this customer's final predicted churn probability
- **Red bars** = features pushing churn risk UP for this customer
- **Blue bars** = features pushing churn risk DOWN
- Bar length = magnitude of that feature's contribution

You can now say: *"This customer is high risk primarily because of X and Y — not because of Z."*  
That's an actionable retention brief.

## 11.4 — Top 3 Risk Reasons Per Customer

Scale the waterfall logic to all customers.  
Output: a table where each row is a customer, with their top 3 churn drivers ranked by SHAP magnitude.

This is what you'd hand to a retention ops team.

In [ ]:
# Build SHAP matrix for test set — rows=customers, cols=features
shap_matrix = pd.DataFrame(
    shap_churn.values,
    columns=[name_map.get(f, f) for f in FEATURES]
)

def top_risk_reasons(row, n=3):
    """Return top N features driving churn risk (positive SHAP only)."""
    positive = row[row > 0].sort_values(ascending=False)
    reasons  = positive.head(n)
    if reasons.empty:
        return ['No strong risk signals'] + ['—'] * (n - 1)
    result = [f'{feat} (+{val:.3f})' for feat, val in reasons.items()]
    while len(result) < n:
        result.append('—')
    return result

reasons_list = shap_matrix.apply(top_risk_reasons, axis=1, result_type='expand')
reasons_list.columns = ['Risk Reason #1', 'Risk Reason #2', 'Risk Reason #3']

# Combine with churn scores
X_test_reset = X_test.reset_index(drop=True)
y_test_reset = y_test.reset_index(drop=True)

df_explained = pd.DataFrame({
    'churn_probability': y_proba_test.round(4),
    'actual_churned':    y_test_reset.values,
}).join(reasons_list)

df_explained = df_explained.sort_values('churn_probability', ascending=False).reset_index(drop=True)

print(f'Explained {len(df_explained)} customers.')
print('\nTop 15 highest-risk customers with churn reasons:\n')
df_explained.head(15)

In [ ]:
# What are the most common #1 risk reasons across all high-risk customers?
high_risk_customers = df_explained[df_explained['churn_probability'] >= 0.70].copy()

# Strip SHAP values from reason strings to get just the feature name
high_risk_customers['top_reason_clean'] = (
    high_risk_customers['Risk Reason #1']
    .str.replace(r' \(\+.*\)', '', regex=True)
)

reason_counts = (
    high_risk_customers['top_reason_clean']
    .value_counts()
    .reset_index()
)
reason_counts.columns = ['Primary Risk Driver', 'Count']
reason_counts['% of High-Risk Customers'] = (
    reason_counts['Count'] / len(high_risk_customers) * 100
).round(1).astype(str) + '%'

print(f'Primary churn drivers across {len(high_risk_customers)} high-risk customers:\n')
reason_counts

In [ ]:
# Visualize: most common primary risk drivers in the high-risk segment
fig, ax = plt.subplots(figsize=(9, 4))

colors = ['#EF4444', '#F97316', '#F59E0B', '#84CC16', '#22C55E',
          '#06B6D4', '#6366F1', '#EC4899', '#A855F7', '#14B8A6']

bars = ax.barh(
    reason_counts['Primary Risk Driver'][::-1],
    reason_counts['Count'][::-1],
    color=colors[:len(reason_counts)][::-1]
)
for bar, val in zip(bars, reason_counts['Count'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(val), va='center', fontsize=9)

ax.set_xlabel('Number of High-Risk Customers', fontsize=10)
ax.set_title('Most Common #1 Churn Driver\n(High-Risk Segment Only)', fontweight='bold', fontsize=12)
ax.set_xlim(0, reason_counts['Count'].max() * 1.12)
plt.tight_layout()
plt.savefig('shap_top_risk_reasons.png', bbox_inches='tight')
plt.show()
print('Saved: shap_top_risk_reasons.png')

## 11.5 — Export SHAP Results

In [ ]:
# Export full explained customer table
df_explained.to_csv('customer_churn_explained.csv', index=False)

# Export SHAP values matrix
shap_matrix.round(5).to_csv('shap_values_matrix.csv', index=False)

print('SHAP exports complete:')
print('  customer_churn_explained.csv  — churn probability + top 3 risk reasons per customer')
print('  shap_values_matrix.csv        — raw SHAP values for all test customers')
print('  shap_summary_beeswarm.png     — global driver chart')
print('  shap_waterfall_high_risk.png  — single customer explanation')
print('  shap_top_risk_reasons.png     — most common risk drivers in high-risk segment')

## 11.6 — Business Framing: What to Say About This

---

### What SHAP adds that feature importance can't

| Feature Importance | SHAP |
|---|---|
| Global only | Global **and** per-customer |
| Tells you *what* matters | Tells you *why this person* is at risk |
| Same output for everyone | Unique explanation per customer |
| Hard to act on | Drives specific retention actions |

---

### How to use this in a real retention workflow

1. Score all customers monthly → assign risk tier
2. For High Risk segment → pull `Risk Reason #1` per customer
3. Route to the right retention action:

| Primary Risk Driver | Retention Action |
|---|---|
| Months Inactive | Re-engagement offer — bonus points for first purchase |
| Low Transactions | Spend challenge — 2x rewards for next 30 days |
| Low Spend | Credit limit review — show them their available headroom |
| No Reward Redemptions | Redemption nudge — "You have X points expiring" |
| Single Product | Cross-sell — checking account, savings, or loan offer |

---

### Key portfolio talking point

> *"I didn't just build a model that predicts churn — I built an explanation layer that tells you exactly why each customer is at risk and what to do about it. That's the difference between a report and a decision-support system."*

---
*Extension built as part of an AI portfolio focused on banking and customer intelligence.*